# Pyrates FEP
## Sample: students stuck between level 5 and level 6

Split out of `02_expert_assignation.ipynb` (its former section 7) - local-only, not tracked in git
(see `.gitignore`). Not tied to the expert assignment in that notebook; run after the data cleaning
pipeline has produced `data/cleaned/cleaned_interaction_traces.csv`.

Targets students who reached at least level 5 but never reached level 7 (i.e. did not complete
level 6), a plateau point worth a closer look. About 20 feedback instances per group (`B`/`C`),
trying for at least one instance from levels 1, 3, 4 and 5 in each group, and preferring games with
at least 3 feedback instances so the sample is concentrated into fewer sessions rather than spread
one-per-game.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Paths relative to the notebook: notebooks/ -> project root -> data/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
INPUT_PATH = DATA_DIR / "cleaned" / "cleaned_interaction_traces.csv"

print("Project root :", PROJECT_ROOT.name)
print("Input        :", INPUT_PATH.relative_to(PROJECT_ROOT), "| exists:", INPUT_PATH.exists())

traces = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False, low_memory=False)
feedback = traces[traces["object_name"] == "ASSISTANT_HELP"]

In [ ]:
PLATEAU_TARGET_LEVELS = [1, 3, 4, 5]
PLATEAU_TARGET_PER_GROUP = 20
PLATEAU_MIN_FEEDBACK_PER_GAME = 5  # preferred: games with at least this many feedback instances

level_id_int = traces["level_id"].astype(int)
max_level_reached = level_id_int.groupby(traces["game_id"]).max()
plateau_games = max_level_reached[(max_level_reached >= 5) & (max_level_reached <= 6)].index

plateau_feedback = feedback[
    (feedback["action_name"] == "RECEIVED") & (feedback["game_id"].isin(plateau_games))
].copy()
plateau_feedback["level_id"] = plateau_feedback["level_id"].astype(int)

print(f"Games that reached level 5 but not level 7: {len(plateau_games)}")
print(f"Feedback instances available in those games: {len(plateau_feedback)}")
print(plateau_feedback.groupby(["group_id", "level_id"]).size().rename("feedback").to_frame())

In [ ]:
def select_plateau_sample(group_df, target_n, target_levels, min_per_game=5, seed=0):
    # shuffle once (deterministic tie-break), so game order below is a fixed random order
    shuffled = group_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    by_game = {game_id: rows["id"].tolist() for game_id, rows in shuffled.groupby("game_id")}
    level_of = dict(zip(shuffled["id"], shuffled["level_id"]))

    # games with >= min_per_game feedback instances are preferred, smallest of them first, so
    # the target is approached closely rather than overshot by a single very large session;
    # games below the threshold are only a fallback if the preferred games can't reach the target
    game_sizes = shuffled.groupby("game_id").size()
    big_games = list(game_sizes[game_sizes >= min_per_game].sort_values(kind="stable").index)
    small_games = [g for g in game_sizes.sort_values(kind="stable").index if g not in set(big_games)]

    def has_level(game_id, level):
        return any(level_of[fid] == level for fid in by_game[game_id])

    # Step 1: which games must be included to guarantee at least one instance per target level?
    # every game ends up included *in full* (see step 2), so a level already covered by an
    # already-required game needs no extra game
    required_games: list[str] = []
    for level in target_levels:
        if any(has_level(g, level) for g in required_games):
            continue
        for game_id in big_games + small_games:
            if has_level(game_id, level):
                required_games.append(game_id)
                break

    # Step 2: add whole games - never a partial game - smallest-first among big games, until
    # their combined feedback reaches the target; a game is either fully in or fully out
    pool = list(required_games)
    total = sum(len(by_game[g]) for g in pool)
    for game_pool in (big_games, small_games):
        for game_id in game_pool:
            if total >= target_n:
                break
            if game_id not in pool:
                pool.append(game_id)
                total += len(by_game[game_id])
        if total >= target_n:
            break

    selected_ids = [feedback_id for game_id in pool for feedback_id in by_game[game_id]]
    return group_df.set_index("id").loc[selected_ids].reset_index()


plateau_sample = pd.concat(
    [
        select_plateau_sample(
            plateau_feedback[plateau_feedback.group_id == group],
            PLATEAU_TARGET_PER_GROUP,
            PLATEAU_TARGET_LEVELS,
            min_per_game=PLATEAU_MIN_FEEDBACK_PER_GAME,
        )
        for group in ["B", "C"]
    ],
    ignore_index=True,
).sort_values(["group_id", "game_id", "level_id"]).reset_index(drop=True)

print(f"Selected {len(plateau_sample)} feedback instances "
      f"(B={int((plateau_sample.group_id == 'B').sum())}, C={int((plateau_sample.group_id == 'C').sum())}) "
      f"- target is {PLATEAU_TARGET_PER_GROUP}/group, but every selected game is included in full, "
      f"so the actual total can run a little over")
print(f"Distinct games represented: {plateau_sample['game_id'].nunique()}")
print("Feedback instances per selected game:")
print(plateau_sample.groupby(["group_id", "game_id"]).size().rename("selected").reset_index().groupby("group_id")["selected"].describe())

print("\nLevel coverage (selected instances):")
print(plateau_sample.groupby(["group_id", "level_id"]).size().rename("selected").to_frame())

for level in PLATEAU_TARGET_LEVELS:
    for group in ["B", "C"]:
        covered = ((plateau_sample.group_id == group) & (plateau_sample.level_id == level)).any()
        if not covered:
            print(f"NOTE: no level {level} feedback available for group {group} in this sample.")

# every selected game_id should carry its *complete* feedback set - not a subset of it
coverage_check = (
    plateau_feedback.groupby("game_id").size().rename("available")
    .to_frame()
    .join(plateau_sample.groupby("game_id").size().rename("selected"), how="right")
)
assert (coverage_check["selected"] == coverage_check["available"]).all(), coverage_check[coverage_check["selected"] != coverage_check["available"]]
print("\nOK: every selected game's feedback is included in full (no partial games).")

plateau_sample[["id", "date", "game_id", "level_id", "group_id", "student_id"]]

In [ ]:
PLATEAU_OUTPUT_PATH = DATA_DIR / "cleaned" / "plateau_feedback_sample.csv"
plateau_sample.to_csv(PLATEAU_OUTPUT_PATH, index=False, encoding="utf-8", lineterminator="\n")
print(f"Written: {PLATEAU_OUTPUT_PATH.relative_to(PROJECT_ROOT)} ({len(plateau_sample)} rows)")